In [ ]:
import json
import pycountry

from emu_renewal.constants import DATA_PATH
from emu_renewal.inputs import get_oxcgrt_data
from emu_renewal.outputs import add_bool_row_to_table
from emu_renewal.selection import get_mob_avail_countries, gather_who_data, find_absent_inds, \
    find_neg_inds, find_outliers, find_nans_repeats, find_pol_countries, get_data_avail_countries

In [ ]:
any_data_avail, summary, _, _, _ = get_data_avail_countries()

In [ ]:
death_data, case_data = gather_who_data(any_data_avail)
no_deaths, no_cases = find_absent_inds(death_data, case_data, summary)
neg_deaths, neg_cases = find_neg_inds(death_data, case_data, summary)
death_outliers, case_outliers = find_outliers(death_data, case_data, summary)
death_nans, case_nans, death_reps, case_reps = find_nans_repeats(death_data, case_data, summary)

In [ ]:
excluded = set(no_deaths + no_cases + neg_deaths + neg_cases + death_nans + case_nans + death_reps + case_reps + death_outliers + case_outliers)
included = [c for c in any_data_avail if c not in excluded]
add_bool_row_to_table(summary, included, "Included")

In [ ]:
summary.index = summary.index.map(lambda iso3: pycountry.countries.lookup(iso3).name)

In [ ]:
mob_included = json.load(open(DATA_PATH / "config/included.json", "r"))

In [ ]:
json.dump(included, open(DATA_PATH / "config/inc_with_pol.json", "w"))

In [ ]:
print("new countries:")
[pycountry.countries.lookup(iso3).name for iso3 in included if iso3 not in mob_included]

In [ ]:
from os import listdir as ls

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.inputs import get_oxcgrt_data

In [ ]:
job_path = OUTPUTS_PATH / "49427755"
all_countries = ls(job_path)
no_oxcgrt_countries = [iso3 for iso3 in all_countries if "oxcgrt" not in ls(job_path / iso3)]
pol_data = get_oxcgrt_data()
pol_avail = [iso3 for iso3 in set(pol_data["CountryCode"]) if iso3 != "RKS"]
rerun_countries = [c for c in included if c not in all_countries]
json.dump(rerun_countries, open(DATA_PATH / "config/rerun_oxcgrt.json", "w"))